In [1]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [2]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")

## 1. Configuração

Por padrão, o gerador procura os notebooks na mesma pasta deste arquivo ou na pasta anterior.  
Ajuste apenas os caminhos se a estrutura do repositório mudar.

In [3]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())

Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [4]:

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


##_______________________________________________________________ "" ____________________________________________________________

##_______________________________________________________________ "" ____________________________________________________________

## FUNIL DE POPULAÇÂO

In [13]:
# ============================================================
# SEPTEMBER PORTFOLIO SNAPSHOT
# Customer universe = QUEUE ∪ WHATSAPP HISTORY
#
# September entrant rule:
#   customer is in QUEUE
#   AND in_collections_since >= 2026-09-01
#   AND has no WhatsApp history
# ============================================================

import pandas as pd

queue = queue.copy()
wa = wa.copy()

queue["in_collections_since"] = pd.to_datetime(
    queue["in_collections_since"],
    errors="coerce"
)

SEP_START = pd.Timestamp("2026-09-01")

# ------------------------------------------------------------
# 1. Customer sets
# ------------------------------------------------------------

queue_ids = set(
    queue["customer_id"]
    .dropna()
    .unique()
)

wa_ids = set(
    wa["customer_id"]
    .dropna()
    .unique()
)

both_ids = queue_ids & wa_ids
wa_only_ids = wa_ids - queue_ids
total_ids = queue_ids | wa_ids

# ------------------------------------------------------------
# 2. TRUE September entrants
# Based on in_collections_since
# ------------------------------------------------------------

sep_entrant_ids = set(
    queue.loc[
        queue["in_collections_since"] >= SEP_START,
        "customer_id"
    ]
    .dropna()
    .unique()
)

# September entrants WITHOUT historical WhatsApp contact
sep_entrant_no_wa_ids = sep_entrant_ids - wa_ids

# ------------------------------------------------------------
# 3. Pre-existing Queue customers without WA history
# Important: these are NOT September entrants
# ------------------------------------------------------------

pre_existing_no_wa_ids = set(
    queue.loc[
        queue["in_collections_since"] < SEP_START,
        "customer_id"
    ]
    .dropna()
    .unique()
) - wa_ids

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

snapshot = pd.DataFrame({
    "population": [
        "Unique customers in Queue",
        "Unique customers in WhatsApp history",
        "September entrants — no WhatsApp history",
        "Pre-existing Queue — no WhatsApp history",
        "Present in both Queue and WhatsApp",
        "WhatsApp history only — not in September Queue",
        "Total unique customers — Queue ∪ WhatsApp",
    ],
    "customers": [
        len(queue_ids),
        len(wa_ids),
        len(sep_entrant_no_wa_ids),
        len(pre_existing_no_wa_ids),
        len(both_ids),
        len(wa_only_ids),
        len(total_ids),
    ]
})

snapshot["pct_total_universe"] = (
    snapshot["customers"] / len(total_ids) * 100
).round(1)

display(snapshot)

,population,customers,pct_total_universe
0,Unique customers in Queue,10658,62.70
1,Unique customers in WhatsApp history,11724,69.00
2,September entrants — no WhatsApp history,5000,29.40
3,Pre-existing Queue — no WhatsApp history,276,1.60
4,Present in both Queue and WhatsApp,5382,31.70
5,WhatsApp history only — not in September Queue,6342,37.30
6,Total unique customers — Queue ∪ WhatsApp,17000,100.00


## 2. Outstanding Balance

In [14]:
# ============================================================
# SEPTEMBER PORTFOLIO SNAPSHOT
# Population + outstanding balance as of 01/09/2026
#
# IMPORTANT:
# outstanding_balance_brl comes ONLY from September Queue.
# WA-only customers have no observable 01/09 balance.
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Customer-level Queue snapshot
# ------------------------------------------------------------

queue_snapshot = (
    queue[
        ["customer_id", "in_collections_since", "outstanding_balance_brl"]
    ]
    .drop_duplicates(subset=["customer_id"])
    .copy()
)

queue_snapshot["in_collections_since"] = pd.to_datetime(
    queue_snapshot["in_collections_since"],
    errors="coerce"
)

queue_balance = (
    queue_snapshot
    .set_index("customer_id")["outstanding_balance_brl"]
)

# ------------------------------------------------------------
# 2. Mutually exclusive populations
# ------------------------------------------------------------

population_groups = {
    "September entrants — no WA history":
        sep_entrant_no_wa_ids,

    "Pre-existing Queue — no WA history":
        pre_existing_no_wa_ids,

    "Queue + WhatsApp history":
        both_ids,

    "WhatsApp history only — not in Sep Queue":
        wa_only_ids,
}

# ------------------------------------------------------------
# 3. Build portfolio snapshot
# ------------------------------------------------------------

rows = []

for population, ids in population_groups.items():

    ids = set(ids)

    # Balance is observable only if customer exists in Queue
    observable_ids = ids & queue_ids

    balance = queue_balance.reindex(list(observable_ids))

    rows.append({
        "population": population,
        "customers": len(ids),
        "pct_total_universe": len(ids) / len(total_ids) * 100,

        "customers_with_sep_balance": balance.notna().sum(),

        "outstanding_balance_sep01_brl":
            balance.sum(min_count=1),

        "avg_outstanding_balance_sep01_brl":
            balance.mean()
    })

portfolio_snapshot = pd.DataFrame(rows)

# ------------------------------------------------------------
# 4. Add TOTAL
# ------------------------------------------------------------

total_queue_balance = queue_balance.sum()

total_row = pd.DataFrame([{
    "population": "TOTAL UNIQUE CUSTOMER UNIVERSE",
    "customers": len(total_ids),
    "pct_total_universe": 100.0,
    "customers_with_sep_balance": queue_balance.notna().sum(),
    "outstanding_balance_sep01_brl": total_queue_balance,
    "avg_outstanding_balance_sep01_brl": queue_balance.mean()
}])

portfolio_snapshot = pd.concat(
    [portfolio_snapshot, total_row],
    ignore_index=True
)

# ------------------------------------------------------------
# 5. Display
# ------------------------------------------------------------

display(
    portfolio_snapshot.style.format({
        "customers": "{:,.0f}",
        "pct_total_universe": "{:.1f}%",
        "customers_with_sep_balance": "{:,.0f}",
        "outstanding_balance_sep01_brl": "R$ {:,.2f}",
        "avg_outstanding_balance_sep01_brl": "R$ {:,.2f}",
    })
)

,population,customers,pct_total_universe,customers_with_sep_balance,outstanding_balance_sep01_brl,avg_outstanding_balance_sep01_brl
0,September entrants — no WA history,"5,000",29.4%,"5,000","R$ 4,284,358.42",R$ 856.87
1,Pre-existing Queue — no WA history,276,1.6%,276,"R$ 218,985.81",R$ 793.43
2,Queue + WhatsApp history,"5,382",31.7%,"5,382","R$ 4,382,663.75",R$ 814.32
3,WhatsApp history only — not in Sep Queue,"6,342",37.3%,0,R$ nan,R$ nan
4,TOTAL UNIQUE CUSTOMER UNIVERSE,"17,000",100.0%,"10,658","R$ 8,886,007.98",R$ 833.74


In [15]:
# ============================================================
# FULL OBSERVED POPULATION SNAPSHOT
# Universe = Queue ∪ WhatsApp history
#
# Outstanding balance:
#   - comes from Queue snapshot (01/09/2026)
#   - customers absent from Queue receive balance = 0
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Queue balance at customer grain
# ------------------------------------------------------------

queue_balance = (
    queue[
        ["customer_id", "outstanding_balance_brl"]
    ]
    .drop_duplicates(subset=["customer_id"])
    .set_index("customer_id")["outstanding_balance_brl"]
)

# ------------------------------------------------------------
# 2. Same mutually exclusive populations
# ------------------------------------------------------------

population_groups = {
    "September entrants — no WhatsApp history":
        sep_entrant_no_wa_ids,

    "Pre-existing Queue — no WhatsApp history":
        pre_existing_no_wa_ids,

    "Present in both Queue and WhatsApp":
        both_ids,

    "WhatsApp history only — not in September Queue":
        wa_only_ids,
}

# ------------------------------------------------------------
# 3. Build snapshot
# ------------------------------------------------------------

rows = []

for population, ids in population_groups.items():

    ids = list(ids)

    # Customers absent from Queue -> balance = 0
    balances = (
        queue_balance
        .reindex(ids)
        .fillna(0)
    )

    rows.append({
        "population": population,
        "customers": len(ids),
        "outstanding_balance_snapshot_2026_09_01_brl":
            balances.sum()
    })

snapshot_17k = pd.DataFrame(rows)

# ------------------------------------------------------------
# 4. Shares
# ------------------------------------------------------------

total_customers = snapshot_17k["customers"].sum()

total_balance = (
    snapshot_17k[
        "outstanding_balance_snapshot_2026_09_01_brl"
    ].sum()
)

snapshot_17k["customer_share_pct"] = (
    snapshot_17k["customers"]
    / total_customers
    * 100
)

snapshot_17k["outstanding_balance_share_pct"] = (
    snapshot_17k["outstanding_balance_snapshot_2026_09_01_brl"]
    / total_balance
    * 100
)

# ------------------------------------------------------------
# 5. Total universe
# ------------------------------------------------------------

total_row = pd.DataFrame([{
    "population": "Total unique customers — Queue ∪ WhatsApp",
    "customers": total_customers,
    "outstanding_balance_snapshot_2026_09_01_brl": total_balance,
    "customer_share_pct": 100.0,
    "outstanding_balance_share_pct": 100.0
}])

snapshot_17k = pd.concat(
    [snapshot_17k, total_row],
    ignore_index=True
)

# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

display(
    snapshot_17k.style.format({
        "customers": "{:,.0f}",
        "outstanding_balance_snapshot_2026_09_01_brl":
            "R$ {:,.2f}",
        "customer_share_pct": "{:.1f}%",
        "outstanding_balance_share_pct": "{:.1f}%"
    })
)

,population,customers,outstanding_balance_snapshot_2026_09_01_brl,customer_share_pct,outstanding_balance_share_pct
0,September entrants — no WhatsApp history,"5,000","R$ 4,284,358.42",29.4%,48.2%
1,Pre-existing Queue — no WhatsApp history,276,"R$ 218,985.81",1.6%,2.5%
2,Present in both Queue and WhatsApp,"5,382","R$ 4,382,663.75",31.7%,49.3%
3,WhatsApp history only — not in September Queue,"6,342",R$ 0.00,37.3%,0.0%
4,Total unique customers — Queue ∪ WhatsApp,"17,000","R$ 8,886,007.98",100.0%,100.0%


In [19]:
# ============================================================
# DISCOUNT-ADJUSTED HISTORICAL BALANCE
#
# Goal:
# First observed balance
# - actual recovered amount
# - contractual discount granted on settled discount offers
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 0. PREPARATION
# ============================================================

wa = wa.copy()

wa["sent_at"] = pd.to_datetime(wa["sent_at"])

wa = wa.sort_values(
    ["customer_id", "sent_at"]
).reset_index(drop=True)


# ============================================================
# 1. BUILD CANONICAL CUSTOMER SNAPSHOT
# ============================================================
# One row per customer.
#
# first_observed_balance_brl:
# balance observed at the customer's first WhatsApp observation
#
# historical_total_recovered_brl:
# sum of actual payments observed in WhatsApp history
# ============================================================

customer_snapshot = (
    wa
    .groupby("customer_id", as_index=False)
    .agg(
        first_observed_balance_brl=(
            "outstanding_balance_brl",
            "first"
        ),
        historical_total_recovered_brl=(
            "amount_paid_brl",
            "sum"
        ),
        first_observed_at=(
            "sent_at",
            "min"
        ),
        last_observed_at=(
            "sent_at",
            "max"
        )
    )
)

customer_snapshot["historical_total_recovered_brl"] = (
    customer_snapshot["historical_total_recovered_brl"]
    .fillna(0)
)


# ============================================================
# 2. IDENTIFY ACTUAL DISCOUNT SETTLEMENTS
# ============================================================
# IMPORTANT:
# 15% is removed ONLY when the discount settlement was actually
# completed.
#
# Receiving a discount_offer alone is NOT enough.
# ============================================================

DISCOUNT_SETTLEMENT_RATE = 0.85
TOLERANCE_BRL = 0.05

wa_discount = wa.copy()

wa_discount["discount_offer_flag"] = (
    wa_discount["template"]
    .astype(str)
    .str.contains(
        "discount_offer",
        case=False,
        na=False
    )
)


# ------------------------------------------------------------
# Expected payment under discount agreement
#
# Full settlement under discount_offer:
# payment ≈ 85% of outstanding balance
# ------------------------------------------------------------

wa_discount["expected_discount_payment_brl"] = (
    wa_discount["outstanding_balance_brl"]
    * DISCOUNT_SETTLEMENT_RATE
)


# ------------------------------------------------------------
# Settlement flag
# ------------------------------------------------------------

wa_discount["discount_settlement_flag"] = (
    wa_discount["discount_offer_flag"]
    &
    wa_discount["amount_paid_brl"].notna()
    &
    (
        wa_discount["amount_paid_brl"]
        - wa_discount["expected_discount_payment_brl"]
    ).abs().le(TOLERANCE_BRL)
)


# ============================================================
# 3. ECONOMIC DISCOUNT GRANTED
# ============================================================
# 15% of balance is written off ONLY when the discounted
# settlement actually occurred.
# ============================================================

wa_discount["discount_granted_brl"] = np.where(
    wa_discount["discount_settlement_flag"],
    wa_discount["outstanding_balance_brl"]
        * (1 - DISCOUNT_SETTLEMENT_RATE),
    0.0
)


# ============================================================
# 4. CUSTOMER-LEVEL DISCOUNT
# ============================================================
# We retain the first observed completed discount settlement
# per customer.
# ============================================================

discount_customer = (
    wa_discount.loc[
        wa_discount["discount_settlement_flag"],
        [
            "customer_id",
            "sent_at",
            "discount_granted_brl"
        ]
    ]
    .sort_values(["customer_id", "sent_at"])
    .groupby("customer_id", as_index=False)
    .first()
    [
        [
            "customer_id",
            "discount_granted_brl"
        ]
    ]
)


# ============================================================
# 5. MERGE DISCOUNT INTO CANONICAL CUSTOMER SNAPSHOT
# ============================================================

customer_snapshot = (
    customer_snapshot
    .drop(
        columns=["discount_granted_brl"],
        errors="ignore"
    )
    .merge(
        discount_customer,
        on="customer_id",
        how="left",
        validate="1:1"
    )
)

customer_snapshot["discount_granted_brl"] = (
    customer_snapshot["discount_granted_brl"]
    .fillna(0)
)


# ============================================================
# 6. DISCOUNT-ADJUSTED HISTORICAL BALANCE
# ============================================================
#
# Economic identity:
#
# Initial balance
# - actual cash recovered
# - contractual discount/write-off
# = estimated remaining economic balance
#
# ============================================================

customer_snapshot["discount_adjusted_balance_brl"] = (
    customer_snapshot["first_observed_balance_brl"]
    - customer_snapshot["historical_total_recovered_brl"]
    - customer_snapshot["discount_granted_brl"]
)

# Avoid tiny floating-point residuals
customer_snapshot["discount_adjusted_balance_brl"] = (
    customer_snapshot["discount_adjusted_balance_brl"]
    .clip(lower=0)
)


# ============================================================
# 7. VALIDATION
# ============================================================

print("=" * 90)
print("DISCOUNT-ADJUSTED HISTORICAL BALANCE")
print("=" * 90)

print(
    f"Customers                          : "
    f"{customer_snapshot['customer_id'].nunique():,}"
)

print(
    f"First observed balance             : "
    f"R$ {customer_snapshot['first_observed_balance_brl'].sum():,.2f}"
)

print(
    f"Actual recovered                   : "
    f"R$ {customer_snapshot['historical_total_recovered_brl'].sum():,.2f}"
)

print(
    f"Discount granted                   : "
    f"R$ {customer_snapshot['discount_granted_brl'].sum():,.2f}"
)

print(
    f"Discount-adjusted remaining balance: "
    f"R$ {customer_snapshot['discount_adjusted_balance_brl'].sum():,.2f}"
)

print()

print(
    f"Customers with discount settlement : "
    f"{(customer_snapshot['discount_granted_brl'] > 0).sum():,}"
)

print("=" * 90)

DISCOUNT-ADJUSTED HISTORICAL BALANCE
Customers                          : 11,724
First observed balance             : R$ 9,965,540.33
Actual recovered                   : R$ 3,459,305.30
Discount granted                   : R$ 37,433.61
Discount-adjusted remaining balance: R$ 6,468,801.92

Customers with discount settlement : 366


### Dos clientes que : WhatsApp history only — not in September Queue ( quantos nao estão na queue de Sep porque quitaram )

In [22]:
# ============================================================
# WA-ONLY FUNNEL — FULL DEBT SETTLEMENT
#
# Population:
#   Customers observed in WhatsApp history
#   but NOT present in September Queue
#
# Full settlement:
#   first observed historical balance
#   - distinct recovered payments
#   - granted discount
#   <= R$ 0.05
# ============================================================

import pandas as pd
import numpy as np

TOLERANCE_BRL = 0.05


# ============================================================
# 1. IDENTIFY CUSTOMER ID COLUMN IN QUEUE
# ============================================================

print("Queue columns:")
print(queue.columns.tolist())

# Assuming both datasets use customer_id.
# If queue uses another name, change only this line.
QUEUE_CUSTOMER_ID = "customer_id"


# ============================================================
# 2. DEFINE WA-ONLY POPULATION
# ============================================================
#
# WA-only =
# customer exists in WhatsApp history
# AND does NOT exist in September Queue
# ============================================================

queue_customer_ids = set(
    queue[QUEUE_CUSTOMER_ID]
    .dropna()
    .unique()
)

wa_customer_ids = set(
    wa["customer_id"]
    .dropna()
    .unique()
)

wa_only_ids = (
    wa_customer_ids
    - queue_customer_ids
)


print("=" * 70)
print("POPULATION RECONCILIATION")
print("=" * 70)

print(
    f"Unique WhatsApp customers : "
    f"{len(wa_customer_ids):,}"
)

print(
    f"Unique September Queue     : "
    f"{len(queue_customer_ids):,}"
)

print(
    f"WA-only customers          : "
    f"{len(wa_only_ids):,}"
)

print("=" * 70)


# ============================================================
# 3. FILTER CUSTOMER SNAPSHOT
# ============================================================

wa_only = (
    customer_snapshot.loc[
        customer_snapshot["customer_id"].isin(wa_only_ids)
    ]
    .copy()
)

print(
    f"WA-only in customer_snapshot: "
    f"{wa_only['customer_id'].nunique():,}"
)


# ============================================================
# 4. FIND CORRECT FIRST-BALANCE COLUMN
# ============================================================
#
# Depending on which version of customer_snapshot you executed,
# the column may be:
#
# first_observed_historical_balance_brl
# OR
# first_observed_balance_brl
# ============================================================

possible_balance_cols = [
    "first_observed_historical_balance_brl",
    "first_observed_balance_brl"
]

balance_col = next(
    (
        col
        for col in possible_balance_cols
        if col in wa_only.columns
    ),
    None
)

if balance_col is None:
    raise KeyError(
        "Could not find first observed balance column. "
        f"Available columns: {wa_only.columns.tolist()}"
    )

print(
    f"Balance column used       : "
    f"{balance_col}"
)


# ============================================================
# 5. CHECK REQUIRED ECONOMIC COLUMNS
# ============================================================

required_cols = [
    "historical_total_recovered_brl",
    "discount_granted_brl"
]

missing_cols = [
    col
    for col in required_cols
    if col not in wa_only.columns
]

if missing_cols:
    raise KeyError(
        f"Missing required columns: {missing_cols}"
    )


# ============================================================
# 6. RECONSTRUCT RESIDUAL DEBT
# ============================================================
#
# Economic identity:
#
# First observed debt
# - actual cash recovered
# - contractual discount/write-off
# = residual debt
#
# ============================================================

wa_only["reconstructed_residual_brl"] = (
    wa_only[balance_col].fillna(0)
    - wa_only["historical_total_recovered_brl"].fillna(0)
    - wa_only["discount_granted_brl"].fillna(0)
).clip(lower=0)


# ============================================================
# 7. FULL SETTLEMENT FLAG
# ============================================================

wa_only["full_settlement_flag"] = (
    wa_only["reconstructed_residual_brl"]
    <= TOLERANCE_BRL
)


# ============================================================
# 8. CUSTOMER COUNTS
# ============================================================

n_total = (
    wa_only["customer_id"]
    .nunique()
)

n_full = (
    wa_only.loc[
        wa_only["full_settlement_flag"],
        "customer_id"
    ]
    .nunique()
)

n_not_full = (
    n_total
    - n_full
)


# ============================================================
# 9. SUMMARY TABLE
# ============================================================

summary_full_settlement = pd.DataFrame({
    "status": [
        "Full debt settlement",
        "Not fully settled"
    ],
    "customers": [
        n_full,
        n_not_full
    ]
})

summary_full_settlement["pct_of_wa_only"] = (
    summary_full_settlement["customers"]
    / n_total
    * 100
)


display(
    summary_full_settlement.style.format({
        "customers": "{:,.0f}",
        "pct_of_wa_only": "{:.1f}%"
    })
)


# ============================================================
# 10. ECONOMIC RECONCILIATION
# ============================================================

initial_balance = (
    wa_only[balance_col]
    .fillna(0)
    .sum()
)

recovered = (
    wa_only["historical_total_recovered_brl"]
    .fillna(0)
    .sum()
)

discount_granted = (
    wa_only["discount_granted_brl"]
    .fillna(0)
    .sum()
)

remaining_balance = (
    wa_only["reconstructed_residual_brl"]
    .fillna(0)
    .sum()
)


print()
print("=" * 70)
print("WA-ONLY — FULL DEBT SETTLEMENT")
print("=" * 70)

print(
    f"Total WA-only customers : "
    f"{n_total:,}"
)

print(
    f"Fully settled           : "
    f"{n_full:,} "
    f"({n_full/n_total:.1%})"
)

print(
    f"Not fully settled       : "
    f"{n_not_full:,} "
    f"({n_not_full/n_total:.1%})"
)

print()

print("-" * 70)
print("ECONOMIC RECONCILIATION")
print("-" * 70)

print(
    f"Initial observed balance : "
    f"R$ {initial_balance:,.2f}"
)

print(
    f"Actual recovered         : "
    f"R$ {recovered:,.2f}"
)

print(
    f"Discount granted         : "
    f"R$ {discount_granted:,.2f}"
)

print(
    f"Residual balance         : "
    f"R$ {remaining_balance:,.2f}"
)

print("=" * 70)


# ============================================================
# 11. SANITY CHECKS
# ============================================================

assert (
    wa_only["customer_id"].nunique()
    == len(wa_only_ids)
), (
    "Mismatch between WA-only IDs and customer_snapshot."
)

assert (
    n_full + n_not_full
    == n_total
)

print("✓ Customer reconciliation passed")
print("✓ Settlement classification passed")

Queue columns:
['customer_id', 'in_collections_since', 'days_past_due_on_2026-09-01', 'outstanding_balance_brl', 'monthly_salary_brl', 'payday_day_of_month', 'n_prior_transactions', 'account_age_months', 'days_since_last_app_login', 'state_uf']
POPULATION RECONCILIATION
Unique WhatsApp customers : 11,724
Unique September Queue     : 10,658
WA-only customers          : 6,342
WA-only in customer_snapshot: 6,342
Balance column used       : first_observed_balance_brl


,status,customers,pct_of_wa_only
0,Full debt settlement,"3,641",57.4%
1,Not fully settled,"2,701",42.6%



WA-ONLY — FULL DEBT SETTLEMENT
Total WA-only customers : 6,342
Fully settled           : 3,641 (57.4%)
Not fully settled       : 2,701 (42.6%)

----------------------------------------------------------------------
ECONOMIC RECONCILIATION
----------------------------------------------------------------------
Initial observed balance : R$ 5,259,885.65
Actual recovered         : R$ 3,136,314.35
Discount granted         : R$ 37,433.61
Residual balance         : R$ 2,086,138.19
✓ Customer reconciliation passed
✓ Settlement classification passed


In [24]:
# ============================================================
# FULL SETTLEMENT TYPE
#
# Split WA-only customers into:
#   1. Full settlement — discount
#   2. Full settlement — regular payment
#   3. Not fully settled
# ============================================================


# ------------------------------------------------------------
# 1. Settlement classification
# ------------------------------------------------------------

wa_only["settlement_type"] = np.select(
    [
        (
            wa_only["full_settlement_flag"]
            & wa_only["discount_granted_brl"].gt(TOLERANCE_BRL)
        ),

        (
            wa_only["full_settlement_flag"]
            & wa_only["discount_granted_brl"].le(TOLERANCE_BRL)
        )
    ],
    [
        "Full settlement — discount",
        "Full settlement — regular payment"
    ],
    default="Not fully settled"
)


# ------------------------------------------------------------
# 2. Summary by settlement type
# ------------------------------------------------------------

settlement_type_summary = (
    wa_only
    .groupby(
        "settlement_type",
        as_index=False
    )
    .agg(
        customers=(
            "customer_id",
            "nunique"
        ),

        first_observed_debt_brl=(
            balance_col,
            "sum"
        ),

        recovered_brl=(
            "historical_total_recovered_brl",
            "sum"
        ),

        discount_granted_brl=(
            "discount_granted_brl",
            "sum"
        ),

        residual_brl=(
            "reconstructed_residual_brl",
            "sum"
        )
    )
)


# ------------------------------------------------------------
# 3. Share of WA-only population
# ------------------------------------------------------------

settlement_type_summary["pct_of_wa_only"] = (
    settlement_type_summary["customers"]
    / n_total
    * 100
)


# ------------------------------------------------------------
# 4. Additional useful metrics
# ------------------------------------------------------------

settlement_type_summary["recovery_rate_pct"] = np.where(
    settlement_type_summary["first_observed_debt_brl"] > 0,

    settlement_type_summary["recovered_brl"]
    / settlement_type_summary["first_observed_debt_brl"]
    * 100,

    np.nan
)


settlement_type_summary["economic_resolution_pct"] = np.where(
    settlement_type_summary["first_observed_debt_brl"] > 0,

    (
        settlement_type_summary["recovered_brl"]
        + settlement_type_summary["discount_granted_brl"]
    )
    / settlement_type_summary["first_observed_debt_brl"]
    * 100,

    np.nan
)


# ------------------------------------------------------------
# 5. Sort for business interpretation
# ------------------------------------------------------------

settlement_order = [
    "Full settlement — regular payment",
    "Full settlement — discount",
    "Not fully settled"
]

settlement_type_summary["settlement_type"] = pd.Categorical(
    settlement_type_summary["settlement_type"],
    categories=settlement_order,
    ordered=True
)

settlement_type_summary = (
    settlement_type_summary
    .sort_values("settlement_type")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

display(
    settlement_type_summary.style.format({
        "customers": "{:,.0f}",
        "pct_of_wa_only": "{:.1f}%",

        "first_observed_debt_brl":
            "R$ {:,.2f}",

        "recovered_brl":
            "R$ {:,.2f}",

        "discount_granted_brl":
            "R$ {:,.2f}",

        "residual_brl":
            "R$ {:,.2f}",

        "recovery_rate_pct":
            "{:.1f}%",

        "economic_resolution_pct":
            "{:.1f}%"
    })
)


# ------------------------------------------------------------
# 7. Console summary
# ------------------------------------------------------------

print("=" * 90)
print("WA-ONLY — SETTLEMENT TYPE")
print("=" * 90)

for _, row in settlement_type_summary.iterrows():

    print(
        f"{str(row['settlement_type']):35s} | "
        f"{row['customers']:>6,.0f} customers | "
        f"{row['pct_of_wa_only']:>5.1f}%"
    )

print("=" * 90)

,settlement_type,customers,first_observed_debt_brl,recovered_brl,discount_granted_brl,residual_brl,pct_of_wa_only,recovery_rate_pct,economic_resolution_pct
0,Full settlement — regular payment,"3,275","R$ 2,652,705.56","R$ 2,652,705.56",R$ 0.00,R$ 0.05,51.6%,100.0%,100.0%
1,Full settlement — discount,366,"R$ 293,186.54","R$ 255,752.88","R$ 37,433.61",R$ 0.50,5.8%,87.2%,100.0%
2,Not fully settled,"2,701","R$ 2,313,993.55","R$ 227,855.91",R$ 0.00,"R$ 2,086,137.64",42.6%,9.8%,9.8%


WA-ONLY — SETTLEMENT TYPE
Full settlement — regular payment   |  3,275 customers |  51.6%
Full settlement — discount          |    366 customers |   5.8%
Not fully settled                   |  2,701 customers |  42.6%


In [25]:
# ============================================================
# WA-ONLY | NOT FULLY SETTLED
# ESTIMATED DPD ON SEP/01
#
# Logic:
# estimated DPD on Sep/01 =
# last observed DPD
# + days between last interaction and Sep/01
# ============================================================

SNAPSHOT_DATE = pd.Timestamp("2026-09-01")


# ------------------------------------------------------------
# 1. Population: WA-only + not fully settled
# ------------------------------------------------------------

not_full_ids = set(
    wa_only.loc[
        ~wa_only["full_settlement_flag"],
        "customer_id"
    ]
)

not_full_history = (
    wa.loc[
        wa["customer_id"].isin(not_full_ids)
    ]
    .dropna(
        subset=[
            "customer_id",
            "sent_at",
            "days_past_due"
        ]
    )
    .sort_values(
        ["customer_id", "sent_at"]
    )
    .copy()
)


# ------------------------------------------------------------
# 2. Last observed interaction per customer
# ------------------------------------------------------------

last_observation = (
    not_full_history
    .groupby(
        "customer_id",
        as_index=False
    )
    .tail(1)
    [
        [
            "customer_id",
            "sent_at",
            "days_past_due"
        ]
    ]
    .rename(columns={
        "sent_at": "last_interaction_at",
        "days_past_due": "last_observed_dpd"
    })
)


# ------------------------------------------------------------
# 3. Days between last interaction and Sep/01
# ------------------------------------------------------------

last_observation["days_until_sep01"] = (
    SNAPSHOT_DATE
    - last_observation["last_interaction_at"].dt.normalize()
).dt.days


# ------------------------------------------------------------
# 4. Estimated DPD on Sep/01
# ------------------------------------------------------------

last_observation["estimated_dpd_sep01"] = (
    last_observation["last_observed_dpd"]
    + last_observation["days_until_sep01"]
)


# ------------------------------------------------------------
# 5. DPD >= 60 on Sep/01
# ------------------------------------------------------------

last_observation["dpd_60_plus_sep01_flag"] = (
    last_observation["estimated_dpd_sep01"] >= 60
)


# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

n_total = len(last_observation)

n_60_plus = (
    last_observation["dpd_60_plus_sep01_flag"]
    .sum()
)

n_under_60 = (
    n_total - n_60_plus
)


summary_sep_dpd = pd.DataFrame({
    "Sep/01 estimated DPD": [
        "DPD >= 60",
        "DPD < 60"
    ],
    "customers": [
        n_60_plus,
        n_under_60
    ]
})

summary_sep_dpd["pct_of_not_fully_settled"] = (
    summary_sep_dpd["customers"]
    / n_total
    * 100
)


display(
    summary_sep_dpd.style.format({
        "customers": "{:,.0f}",
        "pct_of_not_fully_settled": "{:.1f}%"
    })
)


print("=" * 75)
print("WA-ONLY | NOT FULLY SETTLED | ESTIMATED DPD ON SEP/01")
print("=" * 75)

print(
    f"Customers                : {n_total:,}"
)

print(
    f"Estimated DPD >= 60      : "
    f"{n_60_plus:,} "
    f"({n_60_plus/n_total:.1%})"
)

print(
    f"Estimated DPD < 60       : "
    f"{n_under_60:,} "
    f"({n_under_60/n_total:.1%})"
)

print()

print(
    "Estimated DPD Sep/01 — median:",
    f"{last_observation['estimated_dpd_sep01'].median():.0f}"
)

print(
    "Estimated DPD Sep/01 — min:",
    f"{last_observation['estimated_dpd_sep01'].min():.0f}"
)

print(
    "Estimated DPD Sep/01 — max:",
    f"{last_observation['estimated_dpd_sep01'].max():.0f}"
)

,Sep/01 estimated DPD,customers,pct_of_not_fully_settled
0,DPD >= 60,"2,685",99.4%
1,DPD < 60,16,0.6%


WA-ONLY | NOT FULLY SETTLED | ESTIMATED DPD ON SEP/01
Customers                : 2,701
Estimated DPD >= 60      : 2,685 (99.4%)
Estimated DPD < 60       : 16 (0.6%)

Estimated DPD Sep/01 — median: 77
Estimated DPD Sep/01 — min: 15
Estimated DPD Sep/01 — max: 93
